How to apply transformers in text, image, video, audio and time series etc

Transformer model 
- A backbone architecture to represent and extract information from a sequence of embeddings of discrete tokens.
- It takes input of shape (seq_lenght, embedd_size) and outputs tensors of identical shapes
- Ex-1 : Images 

In [ ]:
import argparse
import math
import os
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torchsummary import summary

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """
    
    """
    def __init__(self, embed_dim, ffn_size, num_heads=2, dropout=0.2):
        super(TransformerEncoderBlock, self).__init__()
        # multi-head attention
        self.mhattn = nn.MultiheadAttention(
            embed_dim = embed_dim,
            num_heads = num_heads,
            dropout=dropout,
            batch_first=True,
        )
        # 2. Feed Forward NN
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_size, embed_dim)
        )
        # 3. Layer normalization
        self.input_ln = nn.LayerNorm(embed_dim)
        self.ffn_ln = nn.LayerNorm(embed_dim)

        # forward pass over the input data x
        def forward(self, x):
            y = self.input_ln(x)
            y, _ = self.mhattn(y, y, y),
            y += x

            z = self.ffn_ln(y)
            z = self.ffn(z)
            z += y

            return z

In [ ]:
class TransformerEncoderBlock(nn.Module):
    """Pre-LayerNorm transformer encoder block with MHSA + feedforward network."""
    def __init__(self, embed_dim, ffn_dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.mha = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, ffn_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ffn_dim, embed_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        y = self.ln1(x)
        attn_out, _ = self.mha(y, y, y, need_weights=False)
        x = x + attn_out
        z = self.ln2(x)
        x = x + self.ffn(z)
        return x

In [ ]:
class VisionTransformer(nn.Module):
    """
    
    """
    def __init__(self,
                 img_dim,
                 num_patches,
                 num_classes,
                 embed_dim,
                 ffn_size,
                 num_heads,
                 dropout,
                 num_blocks):
        super(VisionTransformer, self).__init__()

In [ ]:
class VisionTransformer(nn.Module):
    def __init__(
        self,
        image_size=(3, 32, 32),
        patch_grid=(8, 8),
        embed_dim=64,
        ffn_dim=128,
        num_heads=4,
        num_blocks=4,
        num_classes=10,
        dropout=0.1,
        use_sinusoidal_pos=False,
    ):
        super().__init__()
        C, H, W = image_size
        px = H // patch_grid[0]
        py = W // patch_grid[1]
        assert H % patch_grid[0] == 0 and W % patch_grid[1] == 0, "Image dims must be divisible by patch grid"

        self.patch_grid = patch_grid
        self.patch_size = (px, py)
        self.num_patches = patch_grid[0] * patch_grid[1]
        self.num_channels = C
        self.embed_dim = embed_dim

        # linear projection of flattened patch (C * px * py -> embed_dim)
        patch_area = C * px * py
        self.patch_embed = nn.Linear(patch_area, embed_dim)

        # class token
        self.class_token = nn.Parameter(torch.randn(1, 1, embed_dim))

        # positional encoding (sinusoidal or learnable)
        seq_len = self.num_patches + 1
        if use_sinusoidal_pos:
            self.register_buffer('pos_encoding', self._build_sinusoidal_pos_encoding(seq_len, embed_dim))
        else:
            self.pos_embedding = nn.Parameter(torch.randn(1, seq_len, embed_dim))

        # transformer blocks
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim=embed_dim, ffn_dim=ffn_dim, num_heads=num_heads, dropout=dropout)
            for _ in range(num_blocks)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # classification head
        self.head = nn.Linear(embed_dim, num_classes)

    def _build_sinusoidal_pos_encoding(self, seq_len, dim):
        pe = torch.zeros(seq_len, dim)
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2).float() * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)  # (1, seq_len, dim)

    def patchify(self, x: torch.Tensor):
        # x: (B, C, H, W)
        B, C, H, W = x.shape
        px, py = self.patch_size
        gx, gy = self.patch_grid
        # reshape to (B, C, gx, px, gy, py)
        x = x.view(B, C, gx, px, gy, py)
        # move patch dims next to channels then flatten patch content
        # (B, gx, gy, C, px, py)
        x = x.permute(0, 2, 4, 1, 3, 5)
        # flatten patch spatial & channel dims -> (B, num_patches, patch_area)
        x = x.reshape(B, gx * gy, C * px * py)
        return x

    def forward(self, x: torch.Tensor):
        # patchify + embed
        patches = self.patchify(x)  # (B, num_patches, patch_area)
        patches = self.patch_embed(patches)  # (B, num_patches, embed_dim)

        B = patches.size(0)
        class_tok = self.class_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat([class_tok, patches], dim=1)  # (B, seq_len, embed_dim)

        # add positional encoding
        if hasattr(self, 'pos_encoding'):
            x = x + self.pos_encoding.to(x.dtype).to(x.device)
        else:
            x = x + self.pos_embedding

        # transformer
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)

        cls = x[:, 0]  # (B, embed_dim)
        logits = self.head(cls)
        return logits




# ----------------------------
# Optional: attention visualization helper (requires model modification)
# ----------------------------
# Note: The current compact transformer uses nn.MultiheadAttention with need_weights=False in forward to save memory.
# To visualize attention, modify TransformerEncoderBlock to return attention weights (set need_weights=True) and collect them.
# For brevity we include a simple placeholder function here; users may adjust the block if they need attention maps.





In [ ]:
# ----------------------------
# Training utilities
# ----------------------------

def get_dataloaders(batch_size=128, data_dir='./data', num_workers=4):
    mean = (0.4914, 0.4822, 0.4465)
    std = (0.2470, 0.2435, 0.2616)

    train_transforms = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    test_transforms = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])

    train_set = CIFAR10(root=data_dir, train=True, download=True, transform=train_transforms)
    test_set = CIFAR10(root=data_dir, train=False, download=True, transform=test_transforms)

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=num_workers)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=num_workers)
    return train_loader, test_loader


def train_one_epoch(model, device, dataloader, optimizer, epoch, scaler=None):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for X, y in dataloader:
        X = X.to(device)
        y = y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += X.size(0)

    avg_loss = running_loss / total
    acc = correct / total
    print(f"Epoch {epoch}: Train loss {avg_loss:.4f}, Train acc {acc:.4f}")
    return avg_loss, acc


def evaluate(model, device, dataloader):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            logits = model(X)
            loss = F.cross_entropy(logits, y, reduction='sum')
            running_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += X.size(0)
    avg_loss = running_loss / total
    acc = correct / total
    print(f"Eval loss {avg_loss:.4f}, Eval acc {acc:.4f}")
    return avg_loss, acc

def save_checkpoint(state, outdir, name="best.pth"):
    os.makedirs(outdir, exist_ok=True)
    path = os.path.join(outdir, name)
    torch.save(state, path)
    print(f"Saved checkpoint: {path}")



In [ ]:
# ----------------------------
# Main
# ----------------------------



class Config:
    batch_size:int=128,
    epochs:int=10,
    lr:float=3e-4,
    weight_decay:float=0.05,
    data_dir : str='./data',
    output_dir : str = './checkpoints'
    device: str = 'cuda' if torch.cuda.is_available() else 'cpu'
    seed : int = 42


def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def main():
    args = Config()
    set_seed(args.seed)

    device = torch.device(args.device if torch.cuda.is_available() and 'cuda' in args.device else 'cpu')
    print(f"Using device: {device}")

    train_loader, test_loader = get_dataloaders(batch_size=args.batch_size, data_dir=args.data_dir, num_workers=args.num_workers)

    model = VisionTransformer(
        image_size=(3, 32, 32),
        patch_grid=(8, 8),
        embed_dim=128,
        ffn_dim=256,
        num_heads=4,
        num_blocks=6,
        num_classes=10,
        dropout=0.1,
        use_sinusoidal_pos=False,
    )
    model.to(device)

    # optimizer + scheduler
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    total_steps = args.epochs * len(train_loader)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps)

    best_acc = 0.0
    for epoch in range(1, args.epochs + 1):
        train_loss, train_acc = train_one_epoch(model, device, train_loader, optimizer, epoch)
        val_loss, val_acc = evaluate(model, device, test_loader)
        # step scheduler once per epoch (approx)
        scheduler.step()

        # checkpoint best
        if val_acc > best_acc:
            best_acc = val_acc
            save_checkpoint({
                'epoch': epoch,
                'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'val_acc': val_acc,
            }, args.outdir, name=f"vit_cifar10_best.pth")

    print(f"Training finished. Best val acc: {best_acc:.4f}")


main()
